# EDA — ¿Qué explica la duración de un viaje?

**Curso:** Aprendizaje Automático en la nube
**Datos:** `data/processed/viajes_limpio.parquet` — 108.487 viajes de Citi Bike
(Jersey City y Hoboken, julio de 2026), ya limpios.

---

En el notebook 01 dejamos los datos en condiciones y construimos la variable objetivo.
Aquí respondemos la pregunta que sigue: **¿qué hace que un viaje dure lo que dura?**

Cuatro preguntas, en este orden:

1. ¿Cómo se distribuye la duración? (y por qué esa forma condiciona todo lo demás)
2. ¿Quién viaja? — tipo de usuario y tipo de bicicleta
3. ¿Cuándo? — hora del día y día de la semana
4. ¿Qué tan lejos? — la distancia entre estaciones

Al final, la lista de variables que se lleva el modelo — y una que hay que dejar fuera.


## 1. Preparación

Una nota sobre los colores: la paleta anterior no pasaba la verificación de
accesibilidad. El gris y el verde que distinguían los tipos de bicicleta estaban a
una distancia perceptual de 14,5 (el mínimo aceptable es 15) — incluso con visión
normal cuesta separarlos, y con deuteranopia se vuelven el mismo color. Los pares de
abajo sí pasan: se validaron midiendo su separación bajo visión normal y bajo los
tres tipos de daltonismo.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from trips.config import PROCESSED_DATA_PATH, TARGET_COLUMN

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.autolayout"] = True

# Pares validados: cada uno se usa en gráficas distintas, nunca mezclados
AZUL, ROJO = "#3B6EA5", "#E45756"  # tipo de usuario
TEAL, MORADO = "#0E8A79", "#7B52AB"  # tipo de bicicleta
TINTA, SUAVE = "#3A3A3A", "#8C8C8C"  # texto y elementos de contexto

PALETA_BICI = {"classic_bike": TEAL, "electric_bike": MORADO}
NOMBRE_BICI = {"classic_bike": "clásica", "electric_bike": "eléctrica"}
DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]

df = pd.read_parquet(PROCESSED_DATA_PATH)
print(f"{len(df):,} viajes limpios")
df.head()

## 2. Variables derivadas

El archivo no trae ninguna de las variables que realmente explican un viaje: hay que
construirlas. Cuatro, y cada una responde a algo distinto:

- **`hora`** y **`dia_semana`** — el *cuándo*. Un viaje de las 8 de la mañana un martes
  no es la misma clase de viaje que uno del domingo a mediodía.
- **`distancia_km`** — el *cuánto*. Es la distancia en línea recta entre las dos
  estaciones, con la fórmula de **haversine**, que calcula la distancia sobre una
  esfera a partir de latitudes y longitudes. Ojo: es la línea recta, no el recorrido
  real de la bicicleta por las calles, así que siempre subestima.
- **`velocidad_kmh`** — distancia dividida por duración. Es un **diagnóstico**, no una
  variable para el modelo, y en la sección 6 se explica por qué.

Estas cuatro ya viven en `src/trips/features.py` (`add_features`), no aquí: si el
entrenamiento las recalculara por su cuenta, tarde o temprano se le colaría una
diferencia con lo que se exploró en este notebook, y ese tipo de error no avisa.
`velocidad_kmh` es la excepción — es puramente diagnóstica, así que se queda solo en
este notebook y no entra al paquete.

In [ ]:
from trips.features import add_features

df = add_features(df)
# velocidad_kmh es solo diagnóstico exploratorio (fuga de información si entra al
# modelo, ver sección 6.7): se calcula aquí y no vive en trips.features.
df = df.assign(velocidad_kmh=df["distancia_km"] / (df[TARGET_COLUMN] / 60))

df[[TARGET_COLUMN, "distancia_km", "velocidad_kmh"]].describe().round(2)

## 3. La forma de lo que queremos predecir

Antes de buscar qué explica la duración, hay que mirar la duración misma.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

fuera = (df[TARGET_COLUMN] > 60).mean()
ax1.hist(df[TARGET_COLUMN], bins=60, range=(0, 60), color=AZUL, edgecolor="none")
ax1.set_xlabel(f"duración (minutos) — fuera del gráfico: {fuera:.1%} de los viajes")
ax1.set_ylabel("viajes")
ax1.set_title("Escala original", color=TINTA, fontsize=11, loc="left")
ax1.text(
    0.97,
    0.9,
    f"asimetría {df[TARGET_COLUMN].skew():.1f}",
    transform=ax1.transAxes,
    ha="right",
    color=SUAVE,
    fontsize=9,
)

ax2.hist(np.log1p(df[TARGET_COLUMN]), bins=60, color=AZUL, edgecolor="none")
ax2.set_xlabel("log(1 + duración)")
ax2.set_ylabel("viajes")
ax2.set_title("Escala logarítmica", color=TINTA, fontsize=11, loc="left")
ax2.text(
    0.97,
    0.9,
    f"asimetría {np.log1p(df[TARGET_COLUMN]).skew():.2f}",
    transform=ax2.transAxes,
    ha="right",
    color=SUAVE,
    fontsize=9,
)

fig.suptitle(
    "La misma variable, dos escalas", color=TINTA, fontsize=13, x=0.01, ha="left"
)
plt.show()

**Asimetría de 31,6 y curtosis de 1.515.** Para tener una referencia: una campana
normal tiene asimetría 0 y curtosis 3. Lo que estás viendo a la izquierda no es una
distribución con unos cuantos valores extremos, es una distribución que vive casi
entera pegada al origen —el 91,7% de los viajes dura menos de 20 minutos— con una cola
que se estira hasta las 23 horas.

Al aplicar el logaritmo, la asimetría cae a **1,02** y aparece la forma acampanada de
la derecha. Esto no es cosmética: la mayoría de modelos de regresión minimizan el error
cuadrático, y con esta cola un solo viaje de 1.300 minutos pesa más en el
entrenamiento que cientos de viajes normales. **La decisión que sale de esta gráfica es
predecir el logaritmo de la duración, no la duración cruda.**


## 4. Quién viaja

Dos variables describen al usuario: si es miembro o casual, y qué bicicleta tomó.


In [ ]:
for col in ["member_casual", "rideable_type"]:
    tabla = df.groupby(col, observed=True)[TARGET_COLUMN].agg(
        viajes="size", mediana="median", media="mean", p95=lambda s: s.quantile(0.95)
    )
    tabla["%_viajes"] = (tabla["viajes"] / len(df) * 100).round(1)
    print(tabla.round(2).to_string(), "\n")

El tipo de usuario separa con fuerza: **8,6 minutos de mediana los casuales contra
5,8 los miembros**, y en el p95 la distancia se abre todavía más (43,2 contra 20,9). Es
la diferencia entre pasear y desplazarse.

El tipo de bicicleta, en cambio, parece dar casi igual: 6,9 la clásica contra 6,2 la
eléctrica. Menos de un minuto. Pero esa comparación mezcla dos poblaciones muy
distintas, así que hay que separarlas:


In [ ]:
cruce = df.pivot_table(
    index="member_casual",
    columns="rideable_type",
    values=TARGET_COLUMN,
    aggfunc="median",
    observed=True,
).reindex(["member", "casual"])

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(cruce))
ancho = 0.36

for i, (bici, color) in enumerate(PALETA_BICI.items()):
    barras = ax.bar(
        x + (i - 0.5) * (ancho + 0.02),
        cruce[bici].to_numpy(),
        ancho,
        label=NOMBRE_BICI[bici],
        color=color,
    )
    ax.bar_label(barras, fmt="%.1f", padding=3, color=TINTA, fontsize=9)

ax.set_xticks(x, ["miembro", "casual"])
ax.set_ylabel("duración mediana (min)")
ax.set_title(
    "El tipo de bici pesa… solo para los casuales", color=TINTA, fontsize=12, loc="left"
)
ax.legend(frameon=False, ncols=2, loc="upper center", bbox_to_anchor=(0.5, -0.1))
ax.set_ylim(0, cruce.to_numpy().max() * 1.18)
plt.show()

### La mediana no lo cuenta todo

Las barras de arriba comparan medianas, pero una mediana no dice nada sobre la
dispersión: dos grupos con la misma mediana pueden ser uno compacto y otro
desparramado. El diagrama de cajas muestra esa dispersión — la caja abarca la mitad
central del grupo (del percentil 25 al 75) y los bigotes llegan a los percentiles 5 y 95.

Va en **escala logarítmica** por lo mismo de la sección 3: en escala lineal las cajas
quedan aplastadas contra el eje y la gráfica no se lee.


In [ ]:
NOMBRE_BICI = {"classic_bike": "clásica", "electric_bike": "eléctrica"}
datos = df.assign(
    usuario=df["member_casual"].map({"member": "miembro", "casual": "casual"}),
    bici=df["rideable_type"].map(NOMBRE_BICI),
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(
    data=datos,
    x=TARGET_COLUMN,
    y="usuario",
    hue="bici",
    order=["miembro", "casual"],
    hue_order=["clásica", "eléctrica"],
    palette=[TEAL, MORADO],
    whis=(5, 95),
    fliersize=0,
    linewidth=1.1,
    width=0.6,
    ax=ax,
)
ax.set_xscale("log")
ax.set_xlabel("duración (minutos, escala logarítmica) — bigotes: percentiles 5 y 95")
ax.set_ylabel("")
ax.set_title(
    "La caja es la mitad central de cada grupo", color=TINTA, fontsize=12, loc="left"
)
ax.legend(
    frameon=False, ncols=2, loc="upper center", bbox_to_anchor=(0.5, -0.18), title=""
)
plt.show()

Lo que agrega esta gráfica: los casuales en bicicleta clásica no solo tienen la
mediana más alta, tienen **la caja más ancha y el bigote superior más largo**. Son el
grupo más impredecible del conjunto, y por lo tanto el que más le va a costar al
modelo. Los miembros forman cajas estrechas: gente que repite el mismo trayecto.


Ahí está lo que la tabla escondía. Para un **casual**, cambiar de clásica a eléctrica
recorta la mediana de 11,0 a 7,6 minutos: **tres minutos y medio**. Para un **miembro**,
de 5,9 a 5,7: un cuarto de minuto, nada.

Eso se llama una **interacción**: el efecto de una variable depende del valor de otra.
Y tiene una consecuencia práctica para el modelado — un modelo lineal solo la captura si
alguien le escribe explícitamente el término de interacción; los modelos de árboles la
encuentran solos. Es un argumento a favor de probar árboles en la parte 4.

¿Y por qué la eléctrica ayuda tanto a unos y tan poco a otros? La velocidad da la pista:


In [ ]:
con_dist = df[df["distancia_km"] > 0]

print("velocidad implícita (km/h), mediana por tipo de bici:")
print(
    con_dist.groupby("rideable_type", observed=True)["velocidad_kmh"]
    .median()
    .round(2)
    .to_string()
)
print("\ndistancia mediana (km) por usuario y bici:")
print(
    con_dist.pivot_table(
        index="member_casual",
        columns="rideable_type",
        values="distancia_km",
        aggfunc="median",
        observed=True,
    )
    .round(2)
    .to_string()
)

**12,8 km/h la eléctrica contra 9,2 la clásica: un 39% más rápida.** El motor está
haciendo su trabajo. Lo que cambia es a quién le sirve: el miembro que se mueve
quince cuadras hasta la estación del PATH llega igual de rápido con cualquiera de las
dos, porque su viaje es corto y está lleno de semáforos. El casual que se da una vuelta
larga sí nota el motor en el reloj.


## 5. Cuándo se pedalea

Aquí conviene una advertencia sobre gráficas: la tentación es poner el número de viajes
y la duración mediana en la misma gráfica con dos ejes verticales. **No se hace.** Dos
escalas distintas en un mismo marco hacen que las curvas se crucen donde uno decida
poner los límites, y la relación que uno "ve" es un artefacto del dibujo. Dos paneles,
mismo eje horizontal: se comparan igual de bien y no mienten.


In [ ]:
por_hora = df.groupby("hora")[TARGET_COLUMN].agg(viajes="size", mediana="median")

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]}
)

ax1.bar(por_hora.index, por_hora["viajes"], color=SUAVE, width=0.75)
ax1.set_ylabel("viajes")
ax1.set_title(
    "Cuántos viajes empiezan a cada hora", color=TINTA, fontsize=11, loc="left"
)

ax2.plot(por_hora.index, por_hora["mediana"], color=AZUL, lw=2, marker="o", ms=5)
ax2.set_ylabel("duración mediana (min)")
ax2.set_xlabel("hora de inicio")
ax2.set_title("Cuánto duran", color=TINTA, fontsize=11, loc="left")
ax2.set_xticks(range(0, 24, 2))
plt.show()

Los dos paneles cuentan historias opuestas, y ahí está lo interesante.

El **volumen** tiene la forma clásica del desplazamiento al trabajo: un pico a las 8 de
la mañana (7.537 viajes) y otro mucho mayor entre las 5 y las 7 de la tarde (cerca de
9.900 por hora). La **duración**, en cambio, toca su mínimo justo cuando más gente sale:
**4,6 minutos a las 6 de la mañana**, contra 8,6 en la madrugada. Las horas de más
tráfico son las de los viajes más cortos — nadie pasea a las seis de la mañana.

La hora, entonces, no es solo "el momento del día": es un indicador de **para qué** se
usa la bicicleta. Y como es cíclica (las 23 y las 0 son horas vecinas, aunque sus
números estén en extremos opuestos), al modelar hay que decidir si se codifica como
categoría o con senos y cosenos.


In [ ]:
por_dia = df.groupby("dia_semana")[TARGET_COLUMN].median()
colores = [ROJO if d >= 5 else AZUL for d in por_dia.index]

fig, ax = plt.subplots(figsize=(8, 3.6))
barras = ax.bar(
    [DIAS[d] for d in por_dia.index], por_dia.to_numpy(), color=colores, width=0.65
)
ax.bar_label(barras, fmt="%.1f", padding=3, color=TINTA, fontsize=9)
ax.set_ylabel("duración mediana (min)")
ax.set_title(
    "El fin de semana se pedalea distinto", color=TINTA, fontsize=12, loc="left"
)
ax.set_ylim(0, por_dia.max() * 1.2)
plt.show()

**7,7 minutos el sábado y el domingo contra 6,1 los días laborables.** El mismo
patrón de antes: cuando desaparece la obligación, los viajes se alargan.

Un detalle de conteo que vale la pena no tragarse entero: si miras el **número** de
viajes por día, el miércoles parece arrasar con 19.481 contra 12.367 del lunes. Pero
julio de 2026 empezó en miércoles, así que tiene **cinco** miércoles, jueves y viernes,
y solo **cuatro** de los demás días. Normalizando por día, el miércoles baja a 3.896 y
el lunes sube a 3.092: sigue habiendo diferencia, pero mucho menor que la que sugería
el total. Comparar conteos de calendario sin dividir por el número de días es una
trampa clásica.


## 6. La distancia

La variable que uno esperaría que mandara. Pero la correlación depende de cómo se mida:


In [ ]:
print(
    f"distancia mediana: {df['distancia_km'].median():.2f} km   "
    f"|  p95: {df['distancia_km'].quantile(0.95):.2f} km"
)
print(
    f"viajes con distancia 0 (misma estación): {(df['distancia_km'] == 0).sum():,} "
    f"({(df['distancia_km'] == 0).mean():.1%})\n"
)

print("correlación entre duración y distancia")
print(
    f"  Pearson sobre los valores crudos: {con_dist[TARGET_COLUMN].corr(con_dist['distancia_km']):.3f}"
)
print(
    f"  Spearman (sobre los rangos):      {con_dist[TARGET_COLUMN].corr(con_dist['distancia_km'], method='spearman'):.3f}"
)
print(
    f"  Pearson sobre los logaritmos:     {np.log1p(con_dist[TARGET_COLUMN]).corr(np.log1p(con_dist['distancia_km'])):.3f}"
)

**0,19 contra 0,71.** Si te hubieras quedado con el primer número, habrías concluido
que la distancia casi no explica la duración y la habrías descartado como variable.

La diferencia está en qué mide cada coeficiente. **Pearson** mide relación *lineal* sobre
los valores tal cual, y con una cola como la de la sección 3 unos pocos viajes de horas
arrastran el cálculo entero. **Spearman** trabaja sobre los rangos —quién es más largo que
quién— y por eso los extremos no lo distorsionan: el 0,71 dice que la relación monótona
sí es fuerte. Y al aplicar logaritmos a las dos variables, Pearson sube a 0,62: la
relación era clara, solo que no en línea recta.


Vale la pena ver esas correlaciones todas juntas, y con los dos métodos lado a lado:


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

NUMERICAS = [TARGET_COLUMN, "distancia_km", "hora", "dia_semana"]
ETIQUETAS = ["duración", "distancia", "hora", "día"]

# Una correlación es una escala divergente: negativo y positivo son direcciones
# opuestas con un neutro en el cero. Nunca un arcoíris.
diverge = LinearSegmentedColormap.from_list("div", [ROJO, "#F4F4F4", AZUL])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for k, (ax, metodo, titulo) in enumerate(
    zip(
        axes,
        ["pearson", "spearman"],
        [
            "Pearson — relación lineal, valores crudos",
            "Spearman — relación monótona, rangos",
        ],
    )
):
    # Sobre con_dist, los mismos viajes de los coeficientes de arriba
    m = con_dist[NUMERICAS].corr(method=metodo)
    m.index, m.columns = ETIQUETAS, ETIQUETAS
    m = m.iloc[1:, :-1]  # fuera la fila y columna vacías
    mascara = np.triu(np.ones_like(m, dtype=bool), k=1)  # fuera el espejo y la diagonal

    sns.heatmap(
        m,
        ax=ax,
        cmap=diverge,
        vmin=-1,
        vmax=1,
        annot=True,
        fmt=".2f",
        square=True,
        linewidths=3,
        linecolor="white",
        mask=mascara,
        cbar=(k == 1),
        cbar_kws={
            "label": "correlación",
            "shrink": 0.8,
            "ticks": [-1, -0.5, 0, 0.5, 1],
        },
        annot_kws={"color": TINTA, "fontsize": 10},
    )
    ax.set_title(titulo, color=TINTA, fontsize=10, loc="left", pad=10)
    ax.tick_params(colors=TINTA, length=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    ax.grid(False)
plt.show()

Tres lecturas de estas dos matrices:

**La celda duración–distancia cambia de color entre una y otra: 0,19 contra 0,71.** Es el mismo par de
variables y los mismos datos: lo único que cambia es cómo se mide la relación. Si tu
tabla de correlaciones es de Pearson y tus datos tienen una cola como esta, la tabla te
esconde variables útiles.

**`hora` y `dia_semana` salen en casi cero, y no hay que creerles.** Son números que no
son cantidades: la hora 23 no es "mayor" que la 1, es su vecina. Una correlación mide si
al subir una variable sube la otra, y eso no tiene sentido en una escala que da la
vuelta. La gráfica de la sección 5 ya demostró que la hora sí explica la duración, con
una forma de U que ninguna correlación lineal puede capturar. **Un cero en esta matriz
no es permiso para descartar una variable.**

**`velocidad_kmh` no está aquí a propósito.** Correlacionaría casi perfecto con la
duración porque está construida dividiendo por ella. Volvemos a eso en la sección 7.


In [ ]:
muestra = con_dist.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    muestra["distancia_km"],
    muestra[TARGET_COLUMN],
    s=8,
    alpha=0.15,
    color=AZUL,
    edgecolors="none",
)

bins = np.logspace(np.log10(0.05), np.log10(con_dist["distancia_km"].max()), 25)
por_tramo = con_dist.groupby(pd.cut(con_dist["distancia_km"], bins), observed=True)[
    TARGET_COLUMN
].agg(["median", "size"])
por_tramo = por_tramo[
    por_tramo["size"] >= 30
]  # tramos con muy pocos viajes: puro ruido
ax.plot(
    [i.mid for i in por_tramo.index],
    por_tramo["median"].to_numpy(),
    color=ROJO,
    lw=2,
    label="mediana por tramo (≥ 30 viajes)",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("distancia en línea recta (km)")
ax.set_ylabel("duración (min)")
ax.set_title(
    "Cada punto es un viaje; la línea resume la tendencia",
    color=TINTA,
    fontsize=12,
    loc="left",
)
ax.legend(frameon=False, loc="upper left")
plt.show()

En escala logarítmica la tendencia se endereza: de los 200 metros en adelante, la línea
roja sube de forma sostenida.

Fíjate en el extremo izquierdo, donde la línea hace lo contrario: por debajo de 200
metros la duración mediana *sube* hasta pasar los 10 minutos. No es un error de la
gráfica. Son viajes que terminan a la vuelta de donde empezaron después de dar un
paseo largo — la distancia en línea recta los mide como si no se hubieran movido. Es
la misma limitación de la sección anterior vista en su forma más extrema, y explica
por qué esta variable, siendo la mejor que tenemos, no basta por sí sola.
La nube alrededor es real —a igual distancia, un viaje puede durar el triple que otro,
dependiendo de la bici, el semáforo y las ganas— y esa dispersión es, en el fondo, el
margen de error con el que va a tener que vivir el modelo.

Dos advertencias sobre esta variable:

- Es la distancia **en línea recta**, no la recorrida. Siempre subestima, y no
  subestima igual en todas partes: donde hay que rodear un parque o cruzar un puente,
  se equivoca más.
- **4.202 viajes (3,9%) tienen distancia cero** porque vuelven a la estación de origen.
  Son los paseos de ida y vuelta que decidimos conservar en la limpieza. Para el
  modelo son un caso raro: distancia cero y una duración perfectamente normal.


## 7. Qué se lleva el modelo

Lo que este EDA deja decidido:

| Variable | Por qué |
|---|---|
| `distancia_km` | La relación es fuerte (Spearman 0,71), aunque no lineal |
| `member_casual` | Separa dos comportamientos distintos: 8,6 min contra 5,8 |
| `rideable_type` | Poco efecto solo, mucho en interacción con el tipo de usuario |
| `hora` | Indica el propósito del viaje; es cíclica, hay que codificarla con cuidado |
| `dia_semana` / `es_finde` | Fin de semana: 7,7 min contra 6,1 |

Y el objetivo se modela como **`log(1 + duracion_min)`**, por la asimetría de 31,6.

### La variable que hay que dejar fuera

`velocidad_kmh` fue útil para entender el efecto de las eléctricas, pero **no puede
entrar al modelo**: se calcula dividiendo la distancia entre la duración, es decir,
contiene la respuesta. Un modelo que la reciba tendrá un desempeño espectacular en las
pruebas y será inútil en la realidad, porque el día que haya que predecir un viaje que
todavía no ha terminado, esa velocidad no existe.

Eso se llama **fuga de información** (*data leakage*), y es de los errores más caros en
proyectos de machine learning, precisamente porque no se manifiesta como un error: se
manifiesta como un resultado demasiado bueno.

> **Siguiente parte:** mover estas variables derivadas al paquete, entrenar los primeros
> modelos y registrarlos con MLflow.
